# Example 7.8 / 2: Set Branching Priorities

Companion notebook for **7.8 Branching Priorities and Solver Tuning** in the AMPL lab Documentation, section "Set Branching Priorities". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Set Branching Priorities

A branching priority tells the solver which integer variables to branch on first: higher values
first. Declare the `priority` suffix, give values with `let`, and the driver passes them when the
solver supports it; Gurobi reads them under its default `mip:priorities=1`. With the same knapsack
data, priorities equal to each item's weight, then to its value:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
suffix priority IN, integer, >= 0, <= 9999;  # branching priority; higher branches first
set ITEMS;                 # items that can be packed
param value {ITEMS} > 0;   # value of each item
param weight {ITEMS} > 0;  # weight of each item
param volume {ITEMS} > 0;  # volume of each item
param max_weight > 0;      # weight capacity
param max_volume > 0;      # volume capacity
var Take {ITEMS} binary;   # 1 if the item is packed
maximize Total_Value: sum {i in ITEMS} value[i] * Take[i];  # total value packed
subject to Weight: sum {i in ITEMS} weight[i] * Take[i] <= max_weight;  # weight capacity
subject to Volume: sum {i in ITEMS} volume[i] * Take[i] <= max_volume;  # volume capacity

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
items = ["i1", "i2", "i3", "i4", "i5", "i6", "i7", "i8"]
ampl.set["ITEMS"] = items
ampl.param["value"] = dict(zip(items, [25, 47, 44, 18, 33, 48, 40, 50]))
ampl.param["weight"] = dict(zip(items, [23, 7, 24, 5, 20, 13, 22, 12]))
ampl.param["volume"] = dict(zip(items, [11, 27, 20, 22, 22, 20, 17, 25]))
ampl.param["max_weight"] = 63
ampl.param["max_volume"] = 82

In [5]:
# run.py
# presolve, cuts and heuristics off, one thread: the branching order is all that differs
options = "presolve=0 cuts=0 heurfrac=0 threads=1"
for rule in ["none", "weight", "value"]:
    if rule != "none":
        ampl.eval(f"let {{i in ITEMS}} Take[i].priority := {rule}[i];")
    ampl.solve(solver="gurobi", gurobi_options=options, verbose=False)
    print(rule, "|", ampl.obj["Total_Value"].value(), "|",
          ampl.get_value("solve_message").splitlines()[-1])

none | 153.0 | 42 branching nodes
weight | 153.0 | 27 branching nodes
value | 153.0 | 18 branching nodes
